# 🧪 Hunyuan3D v2 — 빠른 파이프라인 테스트
ML 모델 로딩 없이 기하학적 파이프라인만 빠르게 검증합니다.
- **GLB→OBJ 변환** / **remesh** / **UV unwrap** / **OpenCV inpaint** / **GLB 출력**
- 예상 소요: 환경 설치 ~5분 + 테스트 실행 ~30초
- 전체 파이프라인 실행 전 오류 여부를 빠르게 확인하는 용도입니다.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. GitHub 저장소 주소 설정
GITHUB_REPO_URL = "https://github.com/jskim062/image_to_3d.git"
print(f"대상 GitHub Repository: {GITHUB_REPO_URL}")

In [ ]:
# 3. 환경 설치 (메인 노트북과 동일)
import os, glob, sys, shutil

REPO_DIR   = '/kaggle/working/Hunyuan3D-2.1'
CUSTOM_DIR = '/kaggle/working/image_to_3d'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git {REPO_DIR}
else:
    print('[*] Hunyuan3D-2.1 이미 존재')

if os.path.exists(CUSTOM_DIR):
    shutil.rmtree(CUSTOM_DIR)
!git clone {GITHUB_REPO_URL} {CUSTOM_DIR}

for item in ['multiview_utils', 'multiview_pipeline_v2.py', 'test_pipeline_fast.py']:
    src  = os.path.join(CUSTOM_DIR, item)
    dest = os.path.join('/kaggle/working', item)
    if os.path.exists(dest):
        shutil.rmtree(dest) if os.path.isdir(dest) else os.remove(dest)
    if os.path.isdir(src):
        shutil.copytree(src, dest)
    else:
        shutil.copy(src, dest)
print('[*] 소스코드 설치 완료')

%cd {REPO_DIR}/hy3dshape
!pip install -r requirements.txt -q
!pip install fast_simplification opencv-python -q

# custom_rasterizer
_rast_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'custom_rasterizer' in d.lower() and os.path.exists(f'{d}/setup.py')),
    None
)
if _rast_dir:
    !pip install -e {_rast_dir} -q
    print('[*] custom_rasterizer 설치 완료')

# DifferentiableRenderer (테스트에선 실패해도 무방 — OpenCV fallback 검증이 목적)
_renderer_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'differentiablerenderer' in d.lower()),
    None
)
if _renderer_dir and os.path.exists(os.path.join(_renderer_dir, 'setup.py')):
    try:
        !pip install -e {_renderer_dir} -q
        print('[*] DifferentiableRenderer 설치 완료')
    except:
        print('[*] DifferentiableRenderer 컴파일 실패 (테스트에서 OpenCV fallback 검증 예정)')

%cd /kaggle/working
print('환경 설치 완료!')

In [ ]:
# 4. 빠른 파이프라인 테스트 실행
import sys, os, subprocess

print("[*] 기하학적 파이프라인 테스트 시작 (ML 모델 없음)...\n")

_proc = subprocess.Popen(
    [sys.executable, '/kaggle/working/test_pipeline_fast.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
)

for line in _proc.stdout:
    print(line, end='', flush=True)

_proc.wait()
if _proc.returncode == 0:
    print("\n✅ 파이프라인 테스트 통과 — 전체 실행 진행 가능")
else:
    print(f"\n❌ 테스트 실패 (code {_proc.returncode}) — 위 로그를 확인하세요")